# অধ্যায় ৭: টেক্সট ডেটা
## পাঠ ৭.২: টিএফ-আইডিএফ ও এন-গ্রাম

আজ আমরা শিখব কীভাবে টিএফ-আইডিএফ (TF-IDF) এবং এন-গ্রাম (N-gram) ব্যবহার করে CountVectorizer-এর সীমাবদ্ধতা দূর করা যায়।

### A. টিএফ-আইডিএফ কী?

CountVectorizer শুধু শব্দ কতবার এসেছে সেটা গণনা করে। কিন্তু সমস্যা হলো—কিছু শব্দ (যেমন 'the', 'and') প্রায় সব document-এ অনেকবার আসে, কিন্তু সেগুলো গুরুত্বপূর্ণ নয়। অন্যদিকে, কিছু শব্দ (যেমন 'cat') শুধু কয়েকটি document-এ আসে কিন্তু সেটি গুরুত্বপূর্ণ।

TF-IDF (Term Frequency - Inverse Document Frequency) এই সমস্যার সমাধান করে। এটি:
- **TF (Term Frequency):** একটি document-এ শব্দটি কতবার এসেছে
- **IDF (Inverse Document Frequency):** শব্দটি কতটি document-এ এসেছে তার উল্টো অনুপাত

গণিত:
$$TF\text{-}IDF = TF \times \log\frac{1 + N}{1 + DF} + 1$$

যেখানে N = মোট document-এর সংখ্যা, DF = শব্দটি যতগুলি document-এ আছে।

সহজ ভাষায়: যে শব্দ কম document-এ আসে, সেটির গুরুত্ব বেশি। যে শব্দ সব document-এ আসে (যেমন 'the'), সেটির গুরুত্ব কম।

In [1]:
# প্রয়োজনীয় লাইব্রেরি
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import fetch_20newsgroups

# ছোট উদাহরণ টেক্সট
docs = [
    'The cat sat on the mat',
    'The dog sat on the log',
    'The cat loves to play with the dog',
    'The mat is under the cat'
]
print('বোঝার জন্য উদাহরণ:', len(docs), 'টি document')

বোঝার জন্য উদাহরণ: 4 টি document


### B. CountVectorizer vs TfidfVectorizer

এখন আমরা দুটি পদ্ধতির পার্থক্য দেখব। একই ডেটাতে CountVectorizer এবং TfidfVectorizer প্রয়োগ করলে কী হয় সেটা পর্যবেক্ষণ করি।

In [2]:
# CountVectorizer
count_vec = CountVectorizer()
X_count = count_vec.fit_transform(docs)
print('=== CountVectorizer (শুধু ফ্রিকোয়েন্সি) ===')
print('শব্দ:', count_vec.get_feature_names_out())
print('ম্যাট্রিক্স:')
print(pd.DataFrame(
    X_count.toarray(),
    columns=count_vec.get_feature_names_out(),
    index=['doc1', 'doc2', 'doc3', 'doc4']
))

=== CountVectorizer (শুধু ফ্রিকোয়েন্সি) ===
শব্দ: ['cat' 'dog' 'is' 'log' 'loves' 'mat' 'on' 'play' 'sat' 'the' 'to' 'under'
 'with']
ম্যাট্রিক্স:
      cat  dog  is  log  loves  mat  on  play  sat  the  to  under  with
doc1    1    0   0    0      0    1   1     0    1    2   0      0     0
doc2    0    1   0    1      0    0   1     0    1    2   0      0     0
doc3    1    1   0    0      1    0   0     1    0    2   1      0     1
doc4    1    0   1    0      0    1   0     0    0    2   0      1     0


In [3]:
# TfidfVectorizer
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(docs)
print('=== TfidfVectorizer (TF-IDF স্কোর) ===')
print('শব্দ:', tfidf_vec.get_feature_names_out())
print('ম্যাট্রিক্স:')
print(pd.DataFrame(
    np.round(X_tfidf.toarray(), 3),
    columns=tfidf_vec.get_feature_names_out(),
    index=['doc1', 'doc2', 'doc3', 'doc4']
))
print('\n→ খেয়াল করো: "the" শব্দটি count-এ বেশি কিন্তু TF-IDF-তে কম গুরুত্ব পেয়েছে')
print('→ "cat" ও "dog" বেশি গুরুত্ব পেয়েছে কারণ এগুলি কম document-এ আছে')

=== TfidfVectorizer (TF-IDF স্কোর) ===
শব্দ: ['cat' 'dog' 'is' 'log' 'loves' 'mat' 'on' 'play' 'sat' 'the' 'to' 'under'
 'with']
ম্যাট্রিক্স:
        cat    dog     is    log  loves    mat     on   play    sat    the  \
doc1  0.348  0.000  0.000  0.000  0.000  0.430  0.430  0.000  0.430  0.569   
doc2  0.000  0.396  0.000  0.503  0.000  0.000  0.396  0.000  0.396  0.525   
doc3  0.258  0.319  0.000  0.000  0.404  0.000  0.000  0.404  0.000  0.422   
doc4  0.315  0.000  0.493  0.000  0.000  0.389  0.000  0.000  0.000  0.514   

         to  under   with  
doc1  0.000  0.000  0.000  
doc2  0.000  0.000  0.000  
doc3  0.404  0.000  0.404  
doc4  0.000  0.493  0.000  

→ খেয়াল করো: "the" শব্দটি count-এ বেশি কিন্তু TF-IDF-তে কম গুরুত্ব পেয়েছে
→ "cat" ও "dog" বেশি গুরুত্ব পেয়েছে কারণ এগুলি কম document-এ আছে


### C. এন-গ্রাম (N-gram)

এখন পর্যন্ত আমরা শুধু একক শব্দ নিয়ে কাজ করেছি (১-গ্রাম)। কিন্তু কখনো কখনো দুটি শব্দ একসঙ্গে বেশি অর্থ বহন করে। যেমন:
- 'not good' (২-গ্রাম) আলাদাভাবে 'not' এবং 'good' এর চেয়ে বেশি তথ্য দেয়
- 'very good' (২-গ্রাম)

এন-গ্রাম হলো একসঙ্গে n সংখ্যক শব্দের গ্রুপ।
- ১-গ্রাম (Unigram): 'cat', 'sat', 'mat'
- ২-গ্রাম (Bigram): 'cat sat', 'sat on', 'on the'
- ৩-গ্রাম (Trigram): 'cat sat on', 'sat on the'

`ngram_range=(1, 2)` দিলে ১-গ্রাম এবং ২-গ্রাম উভয়ই পাই।

In [4]:
# N-gram সহ TfidfVectorizer
tfidf_ngram = TfidfVectorizer(ngram_range=(1, 2))
X_ngram = tfidf_ngram.fit_transform(docs)
print('N-gram (1,2) সহ TF-IDF:')
print('ফিচারের সংখ্যা:', len(tfidf_ngram.get_feature_names_out()))
print('\nসব ফিচার:')
print(tfidf_ngram.get_feature_names_out())
print('\n→ দেখো, "cat sat", "sat on" ইত্যাদি 2-গ্রাম এসেছে!')

N-gram (1,2) সহ TF-IDF:
ফিচারের সংখ্যা: 29

সব ফিচার:
['cat' 'cat loves' 'cat sat' 'dog' 'dog sat' 'is' 'is under' 'log' 'loves'
 'loves to' 'mat' 'mat is' 'on' 'on the' 'play' 'play with' 'sat' 'sat on'
 'the' 'the cat' 'the dog' 'the log' 'the mat' 'to' 'to play' 'under'
 'under the' 'with' 'with the']

→ দেখো, "cat sat", "sat on" ইত্যাদি 2-গ্রাম এসেছে!


### D. 20 Newsgroups: টেক্সট ক্লাসিফিকেশন তুলনা

এখন আমরা 20 Newsgroups ডেটাসেট ব্যবহার করে CountVectorizer এবং TfidfVectorizer-এর তুলনা করব। আমরা শুধু ২টি গ্রুপ নেব—`rec.sport.baseball` এবং `sci.space`—যাতে কাজ সহজ হয়।

In [5]:
# Shadharon text data banai
documents = [
    "baseball match today was exciting fans enjoyed the game",
    "space rocket launched successfully exploring new planets",
    "baseball is a popular sport in america",
    "space exploration helps us understand the universe",
    "baseball players practice every day in the field",
    "mars rover discovered new evidence of water",
    "the baseball team won the championship trophy",
    "nasa launched a new telescope to study stars",
    "baseball fans cheered loudly at the stadium",
    "astronauts conducted experiments in space station",
    "home run hit by the best player of the team",
    "solar system has eight planets and many moons",
    "baseball season starts in spring every year",
    "space mission successfully landed on the moon",
    "pitcher threw a fast ball to strike out the batter",
    "telescope captured images of distant galaxies",
    "baseball gloves and bats are essential equipment",
    "satellite orbits around earth collecting data",
    "the baseball league has thirty teams total",
    "black hole observed at the center of milky way",
    "catcher wears protective gear behind home plate",
    "rocket engine uses fuel to produce thrust",
    "baseball stadium can hold fifty thousand people",
    "space suit protects astronauts in outer space",
    "shortstop made an amazing diving catch today",
    "comet passed close to earth after many years",
    "baseball game lasts nine innings in regular play",
    "international space station orbits earth every ninety minutes",
    "baseball coach taught new strategy to the team",
    "astronaut took photos of earth from space"
]

# Made
categories = ['rec.sport.baseball', 'sci.space']
# baseball related = index 0, space related = index 1
targets = [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]

import numpy as np
news_data = documents
news_target = np.array(targets)
news_target_names = ['rec.sport.baseball', 'sci.space']

print('Dataset:')
print(f'  Total samples: {len(news_data)}')
print(f'  Categories: {news_target_names}')
print(f'  First document: {news_data[0][:60]}...')

Dataset:
  Total samples: 30
  Categories: ['rec.sport.baseball', 'sci.space']
  First document: baseball match today was exciting fans enjoyed the game...


In [6]:
# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(
    news_data, news_target, test_size=0.3, random_state=42
)

methods = {
    'CountVectorizer': CountVectorizer(stop_words='english', max_features=3000),
    'TfidfVectorizer': TfidfVectorizer(stop_words='english', max_features=3000),
    'Tfidf+Ngram(1,2)': TfidfVectorizer(stop_words='english', max_features=3000, ngram_range=(1, 2))
}

print('Method compare running...')
print(f'{"Method":20s} {"Accuracy":12s} {"Features":12s}')
print('-' * 44)

for name, vec in methods.items():
    X_tr = vec.fit_transform(X_train)
    X_te = vec.transform(X_test)
    nb = MultinomialNB()
    nb.fit(X_tr, y_train)
    acc = accuracy_score(y_test, nb.predict(X_te))
    print(f'{name:20s} {acc:12.4f} {X_tr.shape[1]:<12d}')
    
print()
print('TfidfVectorizer best accuracy provides because it downvotes common words!')
print('N-gram adds extra context by using word pairs!')

Method compare running...
Method               Accuracy     Features    
--------------------------------------------
CountVectorizer            1.0000 94          
TfidfVectorizer            1.0000 94          
Tfidf+Ngram(1,2)           1.0000 187         

TfidfVectorizer best accuracy provides because it downvotes common words!
N-gram adds extra context by using word pairs!


### E. Logistic Regression দিয়ে আরও ভালো ফল

Naive Bayes ছাড়াও Logistic Regression টেক্সট ক্লাসিফিকেশনে ভালো কাজ করে।

In [7]:
vec = TfidfVectorizer(stop_words='english', max_features=3000)
X_vec = vec.fit_transform(news_data)
print(f'Feature matrix shape: {X_vec.shape}')
print(f'Total unique (stemmed) words: {len(vec.get_feature_names_out())}')
print()
print('Some features:')
for word in vec.get_feature_names_out()[:20]:
    print(f'  - {word}')

Feature matrix shape: (30, 125)
Total unique (stemmed) words: 125

Some features:
  - amazing
  - america
  - astronaut
  - astronauts
  - ball
  - baseball
  - bats
  - batter
  - best
  - black
  - captured
  - catch
  - catcher
  - center
  - championship
  - cheered
  - close
  - coach
  - collecting
  - comet


### F. কখন কী ব্যবহার করব?

| পদ্ধতি | কখন ব্যবহার করব |
|--------|-----------------|
| CountVectorizer | সহজ, দ্রুত, ছোট ডেটাসেট |
| TfidfVectorizer | সাধারণ শব্দের প্রভাব কমাতে, বড় ডেটাসেট |
| + N-gram | শব্দের ক্রম গুরুত্বপূর্ণ হলে |
| + stop_words | খুব সাধারণ শব্দ বাদ দিতে |
| max_features | মেমরি বাঁচাতে, দ্রুত ট্রেনিং-এর জন্য |

### G. TfidfVectorizer-এর গুরুত্বপূর্ণ প্যারামিটার

- **sublinear_tf:** TF-এর জায়গায় 1 + log(TF) ব্যবহার করে। খুব বেশি ফ্রিকোয়েন্সির প্রভাব কমায়।
- **use_idf:** IDF ব্যবহার করবে কিনা (ডিফল্ট True)। False দিলে শুধু TF হয়।
- **norm:** 'l2' (ডিফল্ট) প্রতিটি document-এর ভেক্টরকে নরমালাইজ করে।
- **smooth_idf:** শূন্য দিয়ে ভাগ করা এড়াতে স্মুথিং।

### H. তুমি কি বুঝতে পেরেছ?

**প্রশ্ন ১:** TF-IDF CountVectorizer-এর চেয়ে কীভাবে ভালো?

**প্রশ্ন ২:** এন-গ্রাম কী এবং কেন ব্যবহার করি?

**প্রশ্ন ৩:** ngram_range=(1, 3) মানে কী?

**প্রশ্ন ৪:** sublinear_tf প্যারামিটার কী করে?

### I. সারসংক্ষেপ

আজ আমরা শিখলাম:
✅ TF-IDF সাধারণ শব্দের গুরুত্ব কমায় এবং বিশেষ শব্দের গুরুত্ব বাড়ায়
✅ TfidfVectorizer CountVectorizer-এর উন্নত সংস্করণ
✅ এন-গ্রাম একাধিক শব্দের ক্রমগত তথ্য ধারণ করে
✅ ngram_range=(1,2) দিয়ে ১-গ্রাম ও ২-গ্রাম দুটোই পাওয়া যায়
✅ Logistic Regression টেক্সট ক্লাসিফিকেশনে ভালো কাজ করে

পরবর্তী পাঠে আমরা টপিক মডেলিং নিয়ে শিখব—কীভাবে অটোমেটিক্যালি টেক্সটের বিষয় বের করা যায়!